# Lab 2 — Build a curve number for a watershed

**Thirty-five minutes.**

Prepared data is the default. The live path delineates a participant
watershed, then uses Earth Engine to measure land cover and soil jointly.
Both paths finish with the same questions about provenance, unmapped area,
and defensibility.


In [ ]:
# V3 portable setup: local repository, GitHub Pages bundle, or Colab.
from pathlib import Path
import hashlib
import importlib
import importlib.util
import os
import subprocess
import sys
import urllib.request
import zipfile

CNKIT_VERSION = "1.1.0"
BUNDLE_URL = (
    "https://skp703.github.io/cn-workshop-2026/"
    "downloads/cn_workshop_v3_data.zip"
)
BUNDLE_SHA256 = "925861246fe9520c4b7f399227ca6133e60f27a5063a73c9fb6715cd9904780c"


def _is_workshop_root(path):
    return (path / "data" / "sites.csv").exists() and (path / "prepared").exists()


def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/cnkit_workshop"),
    ]
    requested = os.environ.get("CNKIT_WORKSHOP_HOME")
    if requested:
        candidates.insert(0, Path(requested).expanduser())
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_workshop_root(candidate):
            return candidate, "existing workshop folder"

    destination = Path("/content/cnkit_workshop") if Path("/content").exists() else Path.cwd() / ".cnkit_workshop"
    destination.mkdir(parents=True, exist_ok=True)
    archive = destination / "cn_workshop_v3_data.zip"
    print("Downloading the versioned V3 workshop bundle...")
    request = urllib.request.Request(BUNDLE_URL, headers={"User-Agent": "cn-workshop-v3"})
    with urllib.request.urlopen(request, timeout=120) as response, archive.open("wb") as handle:
        handle.write(response.read())
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    if digest != BUNDLE_SHA256:
        raise RuntimeError(
            "Workshop bundle checksum mismatch. Expected %s, received %s. "
            "Delete %s and try again." % (BUNDLE_SHA256, digest, archive)
        )
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(destination)
    if not _is_workshop_root(destination):
        raise RuntimeError("The workshop bundle downloaded but required files are missing.")
    return destination.resolve(), "checksum-verified workshop download"


WORKSHOP_ROOT, DATA_SOURCE = _find_workshop_root()
DATA_DIR = WORKSHOP_ROOT / "data"
PREPARED_DIR = WORKSHOP_ROOT / "prepared"


def _load_cnkit():
    try:
        import cnkit as package
        if getattr(package, "__version__", None) == CNKIT_VERSION:
            return package, "installed package"
    except ImportError:
        pass

    for candidate in [
        WORKSHOP_ROOT / "vendor" / "cnkit.py",
        WORKSHOP_ROOT / "cnkit.py",
        Path.cwd() / "vendor" / "cnkit.py",
        Path.cwd().parent / "vendor" / "cnkit.py",
    ]:
        if candidate.exists():
            spec = importlib.util.spec_from_file_location("cnkit", candidate)
            package = importlib.util.module_from_spec(spec)
            sys.modules["cnkit"] = package
            spec.loader.exec_module(package)
            return package, str(candidate)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit==" + CNKIT_VERSION]
    )
    importlib.invalidate_caches()
    import cnkit as package
    return package, "PyPI"


def activate_full_cnkit():
    """Return the installed package with data, delineation, and GEE modules."""
    global cnkit, CNKIT_SOURCE
    if hasattr(cnkit, "__path__"):
        return cnkit
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit[gee]==" + CNKIT_VERSION]
    )
    for name in [key for key in sys.modules if key == "cnkit" or key.startswith("cnkit.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    cnkit = importlib.import_module("cnkit")
    CNKIT_SOURCE = "PyPI with Earth Engine dependencies"
    return cnkit


cnkit, CNKIT_SOURCE = _load_cnkit()
print("cnkit version:", getattr(cnkit, "__version__", CNKIT_VERSION + " workshop module"))
print("cnkit source :", CNKIT_SOURCE)
print("data source  :", DATA_SOURCE)
print("data folder  :", DATA_DIR)
print("setup complete")


In [ ]:
import json
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cnkit import composite_from_areas

# Change to "accotink_creek" for the second verified reference basin.
PREPARED_WATERSHED = "difficult_run"

# Set True only when the readiness check has already passed.
USE_EARTH_ENGINE = False

# Live-path input: a gage number is easiest. Set GAGE=None and provide
# LAT/LON if you prefer a pour point.
GAGE = "01646000"
LAT, LON = 38.97594, -77.24581


## 1. The prepared tabular route

StreamCat supplies land-cover percentages and Soil Data Access supplies
soil-group percentages. Crossing those two marginal tables assumes they
are independent. We make that assumption explicitly.


In [ ]:
landcover = pd.read_csv(DATA_DIR / "landcover_streamcat.csv")
soils = pd.read_csv(DATA_DIR / "soils_hsg.csv", keep_default_na=False)

def independent_cross(watershed, year):
    lc = landcover[(landcover.watershed == watershed) & (landcover.year == year)].copy()
    sg = soils[soils.watershed == watershed].copy()
    lc["key"], sg["key"] = 1, 1
    crossed = lc.merge(sg, on="key", suffixes=("_lc", "_soil"))
    crossed["area"] = crossed.pct_lc * crossed.pct_soil / 100.0
    return crossed[["nlcd", "hsg", "area"]]

tabular = {}
for condition in ["poor", "fair", "good"]:
    tabular[condition] = composite_from_areas(
        independent_cross(PREPARED_WATERSHED, 2019), condition=condition
    )

pd.DataFrame(tabular).T[
    ["cn_weighted_CN", "cn_weighted_S", "percent_area_unmapped"]
].round(4)


## 2. Recorded live Earth Engine result

The prepared path does not pretend to be live. It reads a result recorded
from an executed Earth Engine notebook, including its asset identifiers,
warning state, and redacted authentication prompt.


In [ ]:
if PREPARED_WATERSHED == "difficult_run":
    recorded = json.loads((PREPARED_DIR / "difficult_run_gee_summary.json").read_text())
    comparison = recorded["curve_number_2019_fair"]
    print("same live raster marginals crossed independently : %.4f" % comparison["marginals_crossed_independently"])
    print("live observed joint distribution                 : %.4f" % comparison["observed_joint_distribution"])
    print("independence assumption                          : %+.4f CN" % comparison["independence_assumption_cn"])
    print("raster soil area with no HSG                     : %.2f %%" % recorded["soils"]["percent_area_no_hsg"])
else:
    print("Accotink has a complete tabular path. Its live-GEE snapshot is not yet recorded; use the live branch or compare tabular results.")


The 1.88-unit difference isolates the independence assumption because
both calculations use the same Earth Engine land-cover and soil
marginals. The older 4.50-unit figure is the range between extreme
feasible pairings of the tabular marginals—a bound, not this measurement.


## 3. Earth Engine application — a selected watershed

Set `USE_EARTH_ENGINE = True` to delineate a selected watershed and
estimate the land-cover–soil joint distribution. The recorded reference
result remains available for direct comparison.


In [ ]:
live = None
live_watershed = None

if USE_EARTH_ENGINE:
    import os
    from getpass import getpass
    activate_full_cnkit()
    import ee
    from cnkit.delineate import watershed_from_gage, watershed_from_point
    from cnkit.gee import Basin, initialise

    project = os.environ.get("CNKIT_EE_PROJECT") or getpass("Earth Engine project ID: ")
    ee.Authenticate()
    initialise(project=project)

    live_watershed = (
        watershed_from_gage(GAGE) if GAGE else watershed_from_point(LAT, LON)
    )
    basin = Basin(live_watershed, project=project)
    joint = basin.joint_landcover_soils(2019, soils="sda")

    lc_marginal = joint.groupby("nlcd", as_index=False)["pct"].sum()
    hsg_marginal = joint.groupby("hsg", as_index=False)["pct"].sum()
    independent = pd.DataFrame(
        [
            {"nlcd": int(a.nlcd), "hsg": s.hsg, "pct": a.pct * s.pct / 100.0}
            for a in lc_marginal.itertuples()
            for s in hsg_marginal.itertuples()
        ]
    )
    kwargs = dict(condition="fair", nlcd_col="nlcd", hsg_col="hsg", area_col="pct")
    observed = composite_from_areas(joint, **kwargs)
    assumed = composite_from_areas(independent, **kwargs)
    live = {
        "area_sqmi": live_watershed.area_sqmi,
        "observed_joint_cn": observed["cn_weighted_CN"],
        "independent_cn": assumed["cn_weighted_CN"],
        "difference_cn": assumed["cn_weighted_CN"] - observed["cn_weighted_CN"],
        "unmapped_pct": observed["percent_area_unmapped"],
        "joint_rows": len(joint),
    }
    print(json.dumps(live, indent=2))
else:
    print("Reference-data analysis active.")
    print("Set USE_EARTH_ENGINE=True to analyze a selected watershed.")


## 4. Put the uncertainty beside the answer


In [ ]:
table = pd.DataFrame(
    {
        condition: {
            "CN weighted by CN": result["cn_weighted_CN"],
            "CN weighted by S": result["cn_weighted_S"],
            "unmapped area, %": result["percent_area_unmapped"],
        }
        for condition, result in tabular.items()
    }
).T
table["condition spread"] = tabular["poor"]["cn_weighted_CN"] - tabular["good"]["cn_weighted_CN"]
table.round(3)


## Report-out

Bring back four values:

1. Composite CN and weighting convention.
2. Poor-to-good condition spread.
3. Area without a mapped hydrologic soil group.
4. Independence correction, measured live or read from the recorded run.

Then answer: **which uncertainty belongs beside the headline CN in a
report?**

**Source anchors:** Annual NLCD Collection 1.2; gNATSGO map-unit raster;
USDA Soil Data Access; EPA StreamCat; NEH-630 Chapters 7 and 9.
